# Real-World Applications of PropFlow

This notebook demonstrates how to model and solve real-world optimization problems as DCOPs using PropFlow.

## Problems We'll Solve

1. **Graph Coloring** - Assign colors to graph nodes so no adjacent nodes share a color
2. **Meeting Scheduling** - Schedule meetings across multiple participants with preferences
3. **Resource Allocation** - Assign limited resources to tasks with dependencies
4. **Sensor Network** - Optimize sensor activation patterns

Each problem shows how to:
- Model the problem as a factor graph
- Design appropriate cost tables
- Choose the right engine/policy
- Interpret and visualize results

In [ ]:
# imports
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import combinations

from propflow import (
    FactorGraph,
    VariableAgent,
    FactorAgent,
    BPEngine,
    DampingEngine,
    MinSumComputator,
    FGBuilder,
)

print("Imports successful!")
np.random.seed(42)

## Problem 1: Graph Coloring

**Problem**: Given a graph, assign colors to nodes such that no two adjacent nodes have the same color.

**DCOP Formulation**:
- Each node is a variable (domain = available colors)
- Each edge creates a binary constraint (factor)
- Cost = 0 if neighbors have different colors, high cost if same color

In [ ]:
# create a graph to color
# we'll use a simple graph structure
def create_graph_coloring_problem(num_nodes=6, num_colors=3, edge_probability=0.4):
    """
    creates a graph coloring problem as a factor graph
    
    args:
        num_nodes: number of nodes in the graph
        num_colors: number of colors available
        edge_probability: probability of edge between any two nodes
    """
    # create random graph
    G = nx.erdos_renyi_graph(num_nodes, edge_probability, seed=42)
    
    # create variable agents (one per node)
    variables = [VariableAgent(name=f"Node_{i}", domain=num_colors) for i in range(num_nodes)]
    
    # create cost table for coloring constraints
    # cost = 0 if different colors (diagonal = 0)
    # cost = 100 if same color (off-diagonal = 100)
    conflict_cost = np.ones((num_colors, num_colors)) * 100
    np.fill_diagonal(conflict_cost, 0)
    
    # create factor for each edge
    factors = []
    edges_dict = {}
    
    for idx, (u, v) in enumerate(G.edges()):
        factor = FactorAgent(
            name=f"f_{u}_{v}",
            domain=num_colors,
            cost_table=conflict_cost.copy()
        )
        factors.append(factor)
        edges_dict[factor] = [variables[u], variables[v]]
    
    # create factor graph
    fg = FactorGraph(
        variables=variables,
        factors=factors,
        edges=edges_dict
    )
    
    return fg, G

# create the problem
coloring_fg, original_graph = create_graph_coloring_problem(
    num_nodes=8,
    num_colors=3,
    edge_probability=0.35
)

print(f"Graph Coloring Problem Created:")
print(f"  Nodes: {len(coloring_fg.variables)}")
print(f"  Edges: {len(coloring_fg.factors)}")
print(f"  Colors available: {coloring_fg.variables[0].domain}")

In [ ]:
# visualize the original graph
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(original_graph, seed=42)
nx.draw_networkx_nodes(original_graph, pos, node_color='lightblue', node_size=700)
nx.draw_networkx_labels(original_graph, pos, font_size=12, font_weight='bold')
nx.draw_networkx_edges(original_graph, pos, width=2, alpha=0.6)
plt.title('Graph to Color', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# solve with damping (cycles in graph may cause oscillations)
coloring_engine = DampingEngine(
    factor_graph=coloring_fg,
    computator=MinSumComputator(),
    damping_factor=0.7
)

coloring_engine.run(max_iter=100)

print(f"\nGraph Coloring Solution:")
print(f"  Converged: {coloring_engine.converged}")
print(f"  Iterations: {coloring_engine.step}")
print(f"  Final cost: {coloring_engine.calculate_global_cost()}")
print(f"\nNode color assignments:")
for node, color in coloring_engine.assignments.items():
    print(f"  {node}: Color {color}")

In [ ]:
# visualize the colored solution
color_map = ['red', 'blue', 'green', 'yellow', 'purple', 'orange']
node_colors = [color_map[coloring_engine.assignments[f"Node_{i}"]] 
               for i in range(len(original_graph.nodes()))]

plt.figure(figsize=(8, 6))
nx.draw_networkx_nodes(original_graph, pos, node_color=node_colors, node_size=700, edgecolors='black', linewidths=2)
nx.draw_networkx_labels(original_graph, pos, font_size=12, font_weight='bold')
nx.draw_networkx_edges(original_graph, pos, width=2, alpha=0.6)
plt.title('Graph Coloring Solution', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

# verify solution
violations = 0
for u, v in original_graph.edges():
    if coloring_engine.assignments[f"Node_{u}"] == coloring_engine.assignments[f"Node_{v}"]:
        violations += 1
        print(f"  ⚠️ Conflict: Node {u} and Node {v} have same color")

if violations == 0:
    print("✅ Valid coloring! No adjacent nodes share colors.")
else:
    print(f"❌ Found {violations} coloring violations")

## Problem 2: Meeting Scheduling with Preferences

**Problem**: Schedule meetings for multiple participants where:
- Each participant has availability constraints
- Some time slots are preferred over others
- Participants involved in same meeting must choose same time

**DCOP Formulation**:
- Variables = participants
- Domain = available time slots
- Unary factors = individual preferences
- Binary factors = coordination constraints (must meet at same time)

In [ ]:
def create_meeting_scheduling_problem():
    """
    creates a meeting scheduling problem
    
    scenario:
    - 5 people: Alice, Bob, Carol, Dave, Eve
    - 4 time slots: 9am, 10am, 11am, 1pm
    - meetings: (Alice, Bob), (Bob, Carol), (Carol, Dave), (Dave, Eve)
    """
    people = ['Alice', 'Bob', 'Carol', 'Dave', 'Eve']
    num_slots = 4
    
    # create variables
    variables = [VariableAgent(name=person, domain=num_slots) for person in people]
    
    # individual preferences (some people prefer certain times)
    # lower cost = more preferred
    preferences = {
        'Alice': np.array([5, 0, 0, 10]),   # prefers 10am, 11am
        'Bob': np.array([0, 5, 10, 15]),    # prefers 9am
        'Carol': np.array([10, 5, 0, 0]),   # prefers 11am, 1pm
        'Dave': np.array([0, 0, 5, 10]),    # prefers 9am, 10am
        'Eve': np.array([15, 10, 5, 0]),    # prefers 1pm
    }
    
    # note: PropFlow doesn't directly support unary factors
    # we'll encode preferences in coordination costs instead
    
    # define meetings (who must meet with whom)
    meetings = [
        ('Alice', 'Bob'),
        ('Bob', 'Carol'),
        ('Carol', 'Dave'),
        ('Dave', 'Eve'),
    ]
    
    # create coordination cost tables
    # diagonal = 0 (same time = good), off-diagonal = 50 (different times = bad)
    coordination_cost = np.ones((num_slots, num_slots)) * 50
    np.fill_diagonal(coordination_cost, 0)
    
    # create factors for meetings
    factors = []
    edges_dict = {}
    
    for person1, person2 in meetings:
        # find variable indices
        var1 = next(v for v in variables if v.name == person1)
        var2 = next(v for v in variables if v.name == person2)
        
        # create modified cost table that includes preferences
        # add preference costs to diagonal
        cost_table = coordination_cost.copy()
        for i in range(num_slots):
            # when both choose time i, add their preference costs
            cost_table[i, i] += preferences[person1][i] + preferences[person2][i]
        
        factor = FactorAgent(
            name=f"meeting_{person1}_{person2}",
            domain=num_slots,
            cost_table=cost_table
        )
        factors.append(factor)
        edges_dict[factor] = [var1, var2]
    
    fg = FactorGraph(variables=variables, factors=factors, edges=edges_dict)
    return fg, people, meetings

scheduling_fg, people, meetings = create_meeting_scheduling_problem()

print(f"Meeting Scheduling Problem:")
print(f"  Participants: {len(people)}")
print(f"  Meetings: {len(meetings)}")
print(f"  Time slots: {scheduling_fg.variables[0].domain}")
print(f"\nMeetings to schedule:")
for m in meetings:
    print(f"  {m[0]} ↔ {m[1]}")

In [ ]:
# solve the scheduling problem
scheduling_engine = DampingEngine(
    factor_graph=scheduling_fg,
    computator=MinSumComputator(),
    damping_factor=0.8
)

scheduling_engine.run(max_iter=100)

time_slots = ['9am', '10am', '11am', '1pm']

print(f"\nScheduling Solution:")
print(f"  Converged: {scheduling_engine.converged}")
print(f"  Final cost: {scheduling_engine.calculate_global_cost():.1f}")
print(f"\nScheduled times:")
for person in people:
    slot = scheduling_engine.assignments[person]
    print(f"  {person:6s}: {time_slots[slot]}")

print(f"\nMeeting times:")
for person1, person2 in meetings:
    slot1 = scheduling_engine.assignments[person1]
    slot2 = scheduling_engine.assignments[person2]
    if slot1 == slot2:
        print(f"  ✅ {person1} & {person2}: {time_slots[slot1]}")
    else:
        print(f"  ❌ {person1} ({time_slots[slot1]}) & {person2} ({time_slots[slot2]}): CONFLICT!")

In [ ]:
# visualize the schedule
fig, ax = plt.subplots(figsize=(10, 6))

# create schedule visualization
for idx, person in enumerate(people):
    slot = scheduling_engine.assignments[person]
    ax.barh(idx, 1, left=slot, height=0.6, 
            color=f"C{slot}", edgecolor='black', linewidth=1.5,
            label=time_slots[slot] if idx == 0 else '')
    ax.text(slot + 0.5, idx, time_slots[slot], 
            ha='center', va='center', fontweight='bold', fontsize=11)

ax.set_yticks(range(len(people)))
ax.set_yticklabels(people, fontsize=12)
ax.set_xticks(range(len(time_slots) + 1))
ax.set_xticklabels([''] + time_slots, fontsize=11)
ax.set_xlabel('Time Slot', fontsize=12)
ax.set_title('Meeting Schedule Solution', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Problem 3: Resource Allocation

**Problem**: Assign tasks to workers where:
- Each worker has different skill levels for different tasks
- Some tasks must be done by workers with compatible skills
- Minimize total cost based on skill mismatch

**DCOP Formulation**:
- Variables = tasks
- Domain = available workers
- Costs = skill mismatch penalties

In [ ]:
def create_resource_allocation_problem():
    """
    assigns 6 tasks to 3 workers based on skills
    """
    tasks = ['Task_A', 'Task_B', 'Task_C', 'Task_D', 'Task_E', 'Task_F']
    num_workers = 3
    
    # create variables (one per task)
    variables = [VariableAgent(name=task, domain=num_workers) for task in tasks]
    
    # skill costs: cost[task][worker] = cost of assigning worker to task
    # lower = better match
    skill_costs = {
        'Task_A': np.array([5, 20, 15]),   # Worker 0 best
        'Task_B': np.array([15, 5, 20]),   # Worker 1 best
        'Task_C': np.array([10, 10, 5]),   # Worker 2 best
        'Task_D': np.array([20, 5, 15]),   # Worker 1 best
        'Task_E': np.array([5, 15, 10]),   # Worker 0 best
        'Task_F': np.array([15, 10, 5]),   # Worker 2 best
    }
    
    # task dependencies: some tasks need workers with compatible skills
    # e.g., Task_A and Task_B should ideally be done by workers with complementary skills
    dependencies = [
        ('Task_A', 'Task_B'),
        ('Task_B', 'Task_C'),
        ('Task_C', 'Task_D'),
        ('Task_D', 'Task_E'),
        ('Task_E', 'Task_F'),
    ]
    
    # create factors
    factors = []
    edges_dict = {}
    
    for task1, task2 in dependencies:
        var1 = next(v for v in variables if v.name == task1)
        var2 = next(v for v in variables if v.name == task2)
        
        # cost table combines skill costs + coordination cost
        cost_table = np.zeros((num_workers, num_workers))
        for w1 in range(num_workers):
            for w2 in range(num_workers):
                # skill mismatch cost
                skill_cost = skill_costs[task1][w1] + skill_costs[task2][w2]
                
                # prefer different workers for dependent tasks (load balancing)
                same_worker_penalty = 10 if w1 == w2 else 0
                
                cost_table[w1, w2] = skill_cost + same_worker_penalty
        
        factor = FactorAgent(
            name=f"dep_{task1}_{task2}",
            domain=num_workers,
            cost_table=cost_table
        )
        factors.append(factor)
        edges_dict[factor] = [var1, var2]
    
    fg = FactorGraph(variables=variables, factors=factors, edges=edges_dict)
    return fg, tasks, num_workers

allocation_fg, tasks, num_workers = create_resource_allocation_problem()

print(f"Resource Allocation Problem:")
print(f"  Tasks: {len(tasks)}")
print(f"  Workers: {num_workers}")
print(f"  Dependencies: {len(allocation_fg.factors)}")

In [ ]:
# solve resource allocation
allocation_engine = DampingEngine(
    factor_graph=allocation_fg,
    computator=MinSumComputator(),
    damping_factor=0.75
)

allocation_engine.run(max_iter=100)

print(f"\nResource Allocation Solution:")
print(f"  Converged: {allocation_engine.converged}")
print(f"  Final cost: {allocation_engine.calculate_global_cost():.1f}")
print(f"\nTask assignments:")

worker_names = ['Worker_0', 'Worker_1', 'Worker_2']
worker_loads = {w: [] for w in range(num_workers)}

for task in tasks:
    worker = allocation_engine.assignments[task]
    worker_loads[worker].append(task)
    print(f"  {task}: {worker_names[worker]}")

print(f"\nWorkload distribution:")
for worker in range(num_workers):
    print(f"  {worker_names[worker]}: {len(worker_loads[worker])} tasks - {worker_loads[worker]}")

In [ ]:
# visualize allocation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# workload distribution
workloads = [len(worker_loads[w]) for w in range(num_workers)]
colors_workers = ['skyblue', 'lightcoral', 'lightgreen']
ax1.bar(worker_names, workloads, color=colors_workers, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Number of Tasks', fontsize=12)
ax1.set_title('Workload Distribution', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# task assignment visualization
task_positions = {task: idx for idx, task in enumerate(tasks)}
for task in tasks:
    worker = allocation_engine.assignments[task]
    ax2.barh(task_positions[task], 1, left=worker, height=0.7,
             color=colors_workers[worker], edgecolor='black', linewidth=1.5)
    ax2.text(worker + 0.5, task_positions[task], worker_names[worker],
             ha='center', va='center', fontweight='bold', fontsize=9)

ax2.set_yticks(list(task_positions.values()))
ax2.set_yticklabels(list(task_positions.keys()), fontsize=10)
ax2.set_xticks(range(num_workers + 1))
ax2.set_xticklabels([''] + worker_names, fontsize=10)
ax2.set_xlabel('Assigned Worker', fontsize=12)
ax2.set_title('Task-to-Worker Assignments', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Problem 4: Sensor Network Activation

**Problem**: In a sensor network:
- Each sensor can be ON (1) or OFF (0)
- Active sensors consume energy (cost)
- Adjacent sensors should not both be active (interference)
- Need minimum coverage: at least some sensors must be active

**DCOP Formulation**:
- Variables = sensors
- Domain = {0: OFF, 1: ON}
- Unary costs = energy consumption
- Binary costs = interference penalties

In [ ]:
def create_sensor_network_problem(num_sensors=10, connectivity=0.3):
    """
    creates a sensor activation problem
    """
    # create random network topology
    G = nx.erdos_renyi_graph(num_sensors, connectivity, seed=42)
    
    # variables: sensor activation state
    variables = [VariableAgent(name=f"Sensor_{i}", domain=2) for i in range(num_sensors)]
    
    # cost table for adjacent sensors
    # [OFF, OFF] = 50 (need coverage, penalty for both off)
    # [OFF, ON]  = 5  (good, one covers)
    # [ON, OFF]  = 5  (good, one covers)
    # [ON, ON]   = 30 (interference penalty)
    interference_cost = np.array([
        [50, 5],   # sensor1=OFF: cost 50 if sensor2=OFF, cost 5 if sensor2=ON
        [5, 30]    # sensor1=ON:  cost 5 if sensor2=OFF, cost 30 if sensor2=ON
    ])
    
    # create factors for edges
    factors = []
    edges_dict = {}
    
    for u, v in G.edges():
        factor = FactorAgent(
            name=f"interference_{u}_{v}",
            domain=2,
            cost_table=interference_cost.copy()
        )
        factors.append(factor)
        edges_dict[factor] = [variables[u], variables[v]]
    
    fg = FactorGraph(variables=variables, factors=factors, edges=edges_dict)
    return fg, G

sensor_fg, sensor_network = create_sensor_network_problem(num_sensors=12, connectivity=0.35)

print(f"Sensor Network Problem:")
print(f"  Sensors: {len(sensor_fg.variables)}")
print(f"  Connections: {len(sensor_fg.factors)}")
print(f"  States per sensor: {sensor_fg.variables[0].domain} (0=OFF, 1=ON)")

In [ ]:
# solve sensor activation
sensor_engine = DampingEngine(
    factor_graph=sensor_fg,
    computator=MinSumComputator(),
    damping_factor=0.8
)

sensor_engine.run(max_iter=150)

print(f"\nSensor Activation Solution:")
print(f"  Converged: {sensor_engine.converged}")
print(f"  Final cost: {sensor_engine.calculate_global_cost():.1f}")

active_sensors = [name for name, state in sensor_engine.assignments.items() if state == 1]
inactive_sensors = [name for name, state in sensor_engine.assignments.items() if state == 0]

print(f"\nActive sensors ({len(active_sensors)}): {', '.join(active_sensors)}")
print(f"Inactive sensors ({len(inactive_sensors)}): {', '.join(inactive_sensors)}")

In [ ]:
# visualize sensor network activation
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(sensor_network, seed=42)

# determine node colors based on activation
node_colors = []
for i in range(len(sensor_network.nodes())):
    if sensor_engine.assignments[f"Sensor_{i}"] == 1:
        node_colors.append('green')  # active
    else:
        node_colors.append('lightgray')  # inactive

# draw network
nx.draw_networkx_nodes(sensor_network, pos, node_color=node_colors, 
                       node_size=800, edgecolors='black', linewidths=2)
nx.draw_networkx_labels(sensor_network, pos, font_size=10, font_weight='bold')
nx.draw_networkx_edges(sensor_network, pos, width=1.5, alpha=0.4)

# add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', edgecolor='black', label='Active (ON)'),
    Patch(facecolor='lightgray', edgecolor='black', label='Inactive (OFF)')
]
plt.legend(handles=legend_elements, loc='upper right', fontsize=11)

plt.title('Sensor Network Activation Solution', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

# check for interference
interference_count = 0
for u, v in sensor_network.edges():
    if sensor_engine.assignments[f"Sensor_{u}"] == 1 and sensor_engine.assignments[f"Sensor_{v}"] == 1:
        interference_count += 1

print(f"\nInterference instances: {interference_count}")
print(f"Coverage: {len(active_sensors)}/{len(sensor_network.nodes())} sensors active")

## Summary

In this tutorial, you learned how to model real-world optimization problems as DCOPs:

1. ✅ **Graph Coloring**: Constraint satisfaction with conflict costs
2. ✅ **Meeting Scheduling**: Coordination with individual preferences
3. ✅ **Resource Allocation**: Task assignment with skill matching
4. ✅ **Sensor Networks**: Binary decisions with coverage/interference tradeoffs

## Key Takeaways

- **Cost table design** is crucial - it encodes your problem's constraints and objectives
- **Graph topology** affects convergence - cycles are harder than trees
- **Damping helps** for most real-world problems with cycles
- **PropFlow is flexible** - you can model many different problem types

## Next Steps

- Try modeling your own domain-specific problems
- Experiment with different cost table designs
- Compare BP with search algorithms (DSA, MGM) for discrete optimization
- Use snapshot analysis to debug convergence issues
- Check the [PropFlow documentation](https://ormullerhahitti.github.io/Belief-Propagation-Simulator/) for advanced features